# 04 — Train HGB với toàn bộ lịch sử và tích hợp Forecast

## tl;dr

Notebook báo cáo toàn bộ quy trình **raw data → nhãn daily → đánh giá theo thời
gian → train cuối → artifact → Forecast UI**. Giữ Restaurants và Day Spas trong
county groups Philadelphia, Nashville, Tampa; không sample check-in hoặc areas.
Model đã chọn từ POC là Histogram Gradient Boosting với Poisson loss; giữ
calendar mean làm benchmark. Không tuning sau khi xem test.

Input dự đoán chỉ gồm **weather, địa điểm, ngành, tháng và weekday từ ngày**.
Activity là nhãn và training weights, không là feature. Kết quả là expected
recorded check-ins / historical cohort business / ngày, không phải doanh thu
hay xác suất thành công của cửa hàng mới.

Chạy **Restart Kernel and Run All** từ kernel `.venv/bin/python`. Notebook tự
kiểm tra nguồn, ghi metrics/figures/`report.md`, train model và xuất bundle vào
`models/location_forecast/`. App đọc bundle này, không train khi render UI.
Output và model giữ local, source notebook không lưu output.

Cấu hình evaluation mới: **train 2010–2019 → validation 2020 → refit
2010–2020 → test 2021**. Catalog evaluation được chọn chỉ từ 2010–2019 và
giữ cố định cho cả hai holdouts. 2020/2021 từng được xem trong lần stress test
trước; đây là backtest khám phá, không phải test chưa từng được nhìn thấy.

Sau evaluation, train bản dùng trong UI trên toàn bộ lịch sử hợp lệ đến
**2022-01-18**. Bảng Results và generated `report.md` chứa số liệu của chính
lần Run All này; metrics của holdouts không phải accuracy đo được của final fit.

Notebook **17 code cells** chạy top-to-bottom. Giữ source notebook không có
outputs; lưu bản executed và report để tái hiện kết quả.


## Context & Methods

1. Xác minh SHA-256 nguồn Yelp/ACIS; đọc đủ business/check-in files trong phạm vi.
2. Kiểm tra ngày biên và missing weather; giữ lịch sử từ **2010-01-01** đến ngày
   trước timestamp cuối trong file. Với snapshot này: **2022-01-18**. Report rõ
   hai check-ins năm 2009 bị loại và ngày 19/01/2022 có thể bị censor; giữ Jan1–18.
3. **Evaluation catalog** chọn bằng activity chỉ trong 2010–2019. Fit 2010–2019,
   validation 2020, refit đến hết 2020, test 2021. Giữ cố định catalog, features
   và parameters; không cập nhật model bằng nhãn 2021 trước khi chấm test.
4. Sau evaluation đã frozen, **final training catalog** chọn từ toàn bộ cửa sổ
   2010–2022-01-18 và train lại HGB trên tất cả row có weather hợp lệ. Các cửa
   hàng/khu vực bổ sung chỉ thuộc catalog cuối; holdout metrics không xác nhận
   accuracy riêng của catalog mở rộng hoặc của bản final fit.
5. Lưu Pipeline + location metadata + observed weather + baseline + manifest,
   kiểm tra load/predict bằng cùng code UI rồi mới publish local bundle.

### Key Assumptions

- Candidate grain giữ source city/state/ZIP; metro là county group có dữ liệu,
  không phải CBSA chính thức. Tọa độ = mean snapshot coordinates trong cohort.
- Minimum **5 cohort businesses và 100 training check-ins** mỗi area/ngành.
  Đây là coverage filter, không phải lấy sample. Denominator cố định theo cohort;
  chưa có lịch sử ngày mở/đóng cửa, nên zero recorded activity không chứng minh
  một business tồn tại/hoạt động trong ngày đó. Snapshot inventory có thể cũ.
- Yelp có timestamps naive; ghép **source date label** với station date, chưa
  xác minh cùng cửa sổ 24h. Không dùng `stars`, `review_count`, `is_open` làm feature.
- Chỉ dùng station numeric finite với flag blank/`T`, °F→°C và inch→mm. Ngày
  thiếu weather bị loại khỏi training/evaluation; không biến missing rain thành 0.
- Historical evaluation dùng observed target-day weather: **conditional
  prediction**, không tự dự báo weather. 2026 có weather nhưng không có activity
  ground truth; thời tiết cho ngày tương lai cần được nhập làm forecast/scenario.
- 2020/2021 đã được xem trong stress test trước: đây là backtest khám phá,
  không phải test untouched. Không tuning theo kết quả mới. Baseline có thể tốt
  hơn HGB; không tuyên bố weather uplift hay kết quả confirmatory.


### 1. Setup

Từ repository root: `uv sync --frozen --group notebooks` và mở Jupyter.
Chạy notebooks 01/02 trước nếu thiếu raw files. Không cần PostgreSQL để train
hoặc dùng trang Demand forecast. Kernel có thể cần quyền bind localhost ports.


In [ ]:
import gc
import hashlib
import importlib.metadata
import json
import re
import shutil
from collections import Counter
from datetime import UTC, datetime
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_poisson_deviance
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from threadpoolctl import threadpool_limits

from sitesense.forecast import FEATURES, LOCATION_COLUMNS, WEATHER, load_bundle, predict_locations
from sitesense.import_data.yelp import categories_from_source

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
pd.set_option("display.max_columns", 14)
pd.set_option("display.max_rows", 15)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True})
thread_limits = threadpool_limits(limits=2)
versions = {
    name: importlib.metadata.version(name)
    for name in [
        "sitesense",
        "numpy",
        "pandas",
        "scikit-learn",
        "nbformat",
        "joblib",
        "threadpoolctl",
    ]
}
display(pd.Series(versions, name="installed version"))

### 2. Config và input contract

Params HGB cố định từ POC; không grid search. Sửa county/category/coverage ở đây
nếu muốn experiment mới. Muốn báo cáo cấu hình khác phải Run All lại.


In [ ]:
METRO_COUNTIES = {
    "Philadelphia": [
        "42017",
        "42029",
        "42045",
        "42091",
        "42101",
        "34005",
        "34007",
        "34015",
        "34033",
        "10003",
    ],
    "Nashville": ["47021", "47037", "47147", "47149", "47165", "47187", "47189"],
    "Tampa": ["12053", "12057", "12101", "12103"],
}
CATEGORIES = ["Restaurants", "Day Spas"]
START = pd.Timestamp("2010-01-01")
INITIAL_CUTOFF = pd.Timestamp("2019-12-31")
VALIDATION_START = pd.Timestamp("2020-01-01")
TEST_START = pd.Timestamp("2021-01-01")
VALIDATION_END = TEST_START - pd.Timedelta(days=1)
TEST_END = TEST_START + pd.DateOffset(years=1) - pd.Timedelta(days=1)
VALIDATION_PERIOD = f"validation_{VALIDATION_START.year}"
TEST_PERIOD = f"test_{TEST_START.year}"
EVALUATION_PROTOCOL = {
    "train_start": str(START.date()),
    "train_end": str(INITIAL_CUTOFF.date()),
    "validation_start": str(VALIDATION_START.date()),
    "validation_end": str(VALIDATION_END.date()),
    "test_refit_end": str(VALIDATION_END.date()),
    "test_start": str(TEST_START.date()),
    "test_end": str(TEST_END.date()),
    "cohort_selection_end": str(INITIAL_CUTOFF.date()),
}
EVALUATION_POLICY = (
    "Train 2010-2019/validation 2020/refit through 2020/test 2021; "
    "fixed pre-validation catalog; exploratory backtest of previously viewed years; "
    "expanded final catalog"
)
FINAL_END = None  # Derived after auditing the source boundary.
MIN_BUSINESSES = 5
MIN_TRAIN_CHECKINS = 100
TOP_K = 3
SEED = 42
SOURCE_NOTEBOOK = ROOT / "notebooks/04_full_history_forecast_training.ipynb"
RUN_DIR = (
    ROOT
    / "data/processed/full_history_forecast"
    / datetime.now(UTC).strftime("run_%Y%m%dT%H%M%S%fZ")
)
assert START <= INITIAL_CUTOFF < VALIDATION_START <= VALIDATION_END < TEST_START <= TEST_END
assert INITIAL_CUTOFF == VALIDATION_START - pd.Timedelta(days=1)
assert MIN_BUSINESSES >= 1
assert len(CATEGORIES) == len(set(CATEGORIES)) and TOP_K >= 1
county_to_metro = {fips: metro for metro, counties in METRO_COUNTIES.items() for fips in counties}
assert len(county_to_metro) == sum(map(len, METRO_COUNTIES.values()))
CAT_FEATURES = ["metro", "category", "calendar_month", "day_of_week"]
NUM_FEATURES = ["latitude", "longitude", *WEATHER]
BASELINE_KEYS = ["area_id", "category", "calendar_month"]
FORBIDDEN_INPUTS = {
    "checkins",
    "rate",
    "n_businesses",
    "train_checkins",
    "lag1",
    "lag12",
    "recent3",
    "trend_years",
}
HGB_PARAMS = dict(
    loss="poisson",
    max_iter=80,
    max_leaf_nodes=15,
    min_samples_leaf=30,
    l2_regularization=1,
    early_stopping=False,
    random_state=SEED,
)
assert FEATURES == CAT_FEATURES + NUM_FEATURES
assert not FORBIDDEN_INPUTS.intersection(FEATURES)
display(pd.DataFrame({"prediction_feature": FEATURES}))

## Data

### 3. Verify source provenance

Manifests ở notebook 01/02 có hashes nguồn đã stage. Dùng complete JSONL files;
review prefix sample không tham gia bài toán này. Kiểm tra hash trước mọi phép tính.


In [ ]:
paths = {
    "business": ROOT / "data/raw/yelp/business.jsonl",
    "checkin": ROOT / "data/raw/yelp/checkin.jsonl",
    "yelp_manifest": ROOT / "data/raw/yelp/staging_manifest.json",
    "mapping": ROOT / "data/raw/climate/yelp_scope/business_climate_mapping.csv",
    "scope_manifest": ROOT / "data/raw/climate/yelp_scope/scope_manifest.json",
    "observations": ROOT / "data/raw/climate/yelp_stations/observations.csv.gz",
    "station_manifest": ROOT / "data/raw/climate/yelp_stations/manifest.json",
}
for path in paths.values():
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path.relative_to(ROOT)}. Run notebooks 01 and 02.")


def sha256(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


source_hashes = {name: sha256(path) for name, path in paths.items()}
yelp_manifest = json.loads(paths["yelp_manifest"].read_text())
station_manifest = json.loads(paths["station_manifest"].read_text())
scope_manifest = json.loads(paths["scope_manifest"].read_text())
assert station_manifest["status"] == "complete"
for name in ["business", "checkin"]:
    assert source_hashes[name] == yelp_manifest["staged_sha256"][name]
    assert paths[name].stat().st_size == yelp_manifest["staged_bytes"][name]
assert source_hashes["observations"] == station_manifest["csv"]["sha256"]
expected_mapping = next(
    item["sha256"]
    for item in scope_manifest["outputs"]
    if item["path"].endswith("/business_climate_mapping.csv")
)
assert source_hashes["mapping"] == expected_mapping
source_audit = pd.DataFrame(
    [
        {"source": name, "path": str(path.relative_to(ROOT)), "sha256": source_hashes[name]}
        for name, path in paths.items()
    ]
)
display(source_audit)
RUN_DIR.mkdir(parents=True, exist_ok=False)
(RUN_DIR / "figures").mkdir()

### 4. Scope ngành và địa điểm

Giữ tất cả business-category rows hợp lệ trong county groups; không lấy top-N
hoặc sample areas. ZIP/coords invalid được ghi vào coverage funnel.


In [ ]:
mapping = pd.read_csv(paths["mapping"], dtype=str, keep_default_na=False)
assert mapping["business_id"].is_unique and len(mapping) == scope_manifest["businesses"]
mapping["metro"] = mapping["county_fips"].map(county_to_metro)
metro_mapping = mapping[mapping["metro"].notna() & mapping["status"].eq("mapped")]
lookup = metro_mapping.set_index("business_id")
records = []
with paths["business"].open() as handle:
    for line in handle:
        item = json.loads(line)
        business_id = item["business_id"]
        if business_id not in lookup.index:
            continue
        labels = categories_from_source(item.get("categories"))
        for category in CATEGORIES:
            if category in labels:
                place = lookup.loc[business_id]
                city = " ".join(item["city"].split()).casefold()
                state = item["state"].strip().upper()
                zip_code = item["postal_code"].strip()
                records.append(
                    {
                        "business_id": business_id,
                        "metro": place["metro"],
                        "city": city,
                        "state": state,
                        "zip": zip_code,
                        "category": category,
                        "latitude": float(item["latitude"]),
                        "longitude": float(item["longitude"]),
                        "station_id": place["station_id"],
                        "station_distance_km": float(place["station_distance_km"]),
                    }
                )
businesses = pd.DataFrame(records)
assert not businesses.empty, "No matching category/metro businesses."
assert not businesses.duplicated(["business_id", "category"]).any()
excluded_invalid_zip = int((~businesses["zip"].str.fullmatch(r"[0-9]{5}")).sum())
valid_coordinates = (
    np.isfinite(businesses[["latitude", "longitude"]]).all(axis=1)
    & businesses["latitude"].between(-90, 90)
    & businesses["longitude"].between(-180, 180)
)
excluded_invalid_coordinates = int((~valid_coordinates).sum())
businesses = businesses[businesses["zip"].str.fullmatch(r"[0-9]{5}") & valid_coordinates].copy()
businesses["area_id"] = businesses[["metro", "state", "city", "zip"]].agg("|".join, axis=1)
businesses["group_id"] = businesses["area_id"] + "|" + businesses["category"]
display(
    businesses.groupby(["metro", "category"]).agg(
        snapshot_businesses=("business_id", "size"), source_areas=("area_id", "nunique")
    )
)
print(f"Excluded business/category rows with invalid ZIP: {excluded_invalid_zip:,}")
print(f"Excluded invalid coordinate rows: {excluded_invalid_coordinates:,}")

### 5. Audit toàn bộ timestamp file và chọn ngày biên

Stream đầy đủ check-in file để đếm coverage và activity dùng chọn cohort. Không
materialize từng event thành business×day dataframe. Final end bỏ ngày cuối file
như một conservative boundary policy; không tạo zero-label sau cutoff nguồn.


In [ ]:
selected_ids = set(businesses["business_id"])
initial_counts, available_counts = Counter(), Counter()
source_day_counts = Counter()
last_business_day = {}
last_business_day_counts = {}
seen_ids = set()
source_first, source_last = "9999", ""
source_records, source_tokens = 0, 0
pattern = re.compile(r"[0-9]{4}-[0-9]{2}-[0-9]{2} [0-9:]{8}")
start_label = START.strftime("%Y-%m-%d")
initial_label = INITIAL_CUTOFF.strftime("%Y-%m-%d")
with paths["checkin"].open() as handle:
    for line in handle:
        item = json.loads(line)
        business_id = item["business_id"]
        assert business_id not in seen_ids
        seen_ids.add(business_id)
        source_records += 1
        timestamps = item["date"].split(", ")
        assert all(pattern.fullmatch(timestamp) for timestamp in timestamps)
        source_tokens += len(timestamps)
        source_first = min(source_first, min(timestamps))
        source_last = max(source_last, max(timestamps))
        for timestamp in timestamps:
            source_day_counts[timestamp[:10]] += 1
        if business_id not in selected_ids:
            continue
        days = [timestamp[:10] for timestamp in timestamps]
        maximum_day = max(days)
        last_business_day[business_id] = maximum_day
        last_business_day_counts[business_id] = days.count(maximum_day)
        available_counts[business_id] = sum(day >= start_label for day in days)
        initial_counts[business_id] = sum(start_label <= day <= initial_label for day in days)
source_last_date = pd.Timestamp(source_last[:10])
FINAL_END = source_last_date - pd.Timedelta(days=1)
assert TEST_END <= FINAL_END
final_counts = available_counts.copy()
for business_id, last_day in last_business_day.items():
    if last_day == source_last[:10]:
        final_counts[business_id] -= last_business_day_counts[business_id]
initial_counts = Counter({key: value for key, value in initial_counts.items() if value > 0})
final_counts = Counter({key: value for key, value in final_counts.items() if value > 0})
source_audit_counts = {
    "source_checkin_records": source_records,
    "source_timestamp_tokens": source_tokens,
    "first_timestamp": source_first,
    "last_timestamp": source_last,
    "training_start": str(START.date()),
    "final_training_end": str(FINAL_END.date()),
    "global_events_before_start": sum(
        count for day, count in source_day_counts.items() if day < start_label
    ),
    "global_events_on_excluded_last_day": source_day_counts[source_last[:10]],
    "valid_scope_business_category_rows": len(businesses),
    "unique_scoped_businesses": businesses.business_id.nunique(),
}
display(pd.Series(source_audit_counts, name="source audit"))
print("Final training includes partial January through", FINAL_END.date())

### 6. Hai cohort có mục đích riêng

Evaluation cohort dùng activity trước validation; final cohort dùng activity
trong toàn bộ final training window. Không gán holdout metric của evaluation
catalog cho các area/business mới ở catalog cuối. Station dominant và coordinates
được cố định theo mỗi cohort; count chỉ dùng coverage/nhãn/weights.


In [ ]:
def select_cohort(activity_counts, label):
    active = businesses[businesses["business_id"].isin(activity_counts)].copy()
    active["train_checkins"] = active["business_id"].map(activity_counts)
    statistics = active.groupby("group_id").agg(
        n_businesses=("business_id", "size"), train_checkins=("train_checkins", "sum")
    )
    eligible = statistics[
        statistics.n_businesses.ge(MIN_BUSINESSES)
        & statistics.train_checkins.ge(MIN_TRAIN_CHECKINS)
    ]
    cohort = active[active.group_id.isin(eligible.index)].copy()
    station_counts = cohort.groupby(["group_id", "station_id"]).size().rename("station_businesses")
    stations = (
        station_counts.reset_index()
        .sort_values(
            ["group_id", "station_businesses", "station_id"], ascending=[True, False, True]
        )
        .drop_duplicates("group_id")
        .set_index("group_id")
    )
    area_table = cohort.groupby("group_id")[
        ["area_id", "metro", "city", "state", "zip", "category"]
    ].first()
    area_table = area_table.join(cohort.groupby("group_id")[["latitude", "longitude"]].mean())
    area_table = area_table.join(eligible).join(stations)
    summary = (
        cohort.groupby(["metro", "category"])
        .agg(
            area_category_groups=("group_id", "nunique"),
            business_category_rows=("business_id", "size"),
            training_checkins=("train_checkins", "sum"),
        )
        .reset_index()
    )
    summary["catalog"] = label
    funnel = {
        "catalog": label,
        "scope_rows": len(businesses),
        "no_activity_in_selection_window": len(businesses) - len(active),
        "low_area_coverage_rows": len(active) - len(cohort),
        "cohort_business_category_rows": len(cohort),
        "cohort_unique_businesses": cohort.business_id.nunique(),
        "area_category_groups": len(area_table),
        "dominant_stations": area_table.station_id.nunique(),
    }
    assert not cohort.empty and not cohort.duplicated(["business_id", "category"]).any()
    return cohort, area_table, summary, funnel


evaluation_cohort, evaluation_areas, eval_summary, eval_funnel = select_cohort(
    initial_counts, "evaluation"
)
final_cohort, final_areas, final_summary, final_funnel = select_cohort(final_counts, "final")
assert set(evaluation_cohort.business_id).issubset(set(final_cohort.business_id))
cohort_comparison = pd.concat([eval_summary, final_summary], ignore_index=True)
funnel = pd.DataFrame([eval_funnel, final_funnel])
display(funnel)
display(cohort_comparison)

### 7. Daily station observations và coverage policy

Giữ raw station date, không gom tháng; finite numeric với flag blank/`T`. Dùng
mọi selected station report kể cả 2026 cho inference, nhưng training join chỉ
đến activity cutoff. Không dùng dữ liệu tương lai để fit preprocessing.


In [ ]:
observations = pd.read_csv(
    paths["observations"],
    usecols=["station_id", "date", "variable", "value", "unit", "flag"],
    dtype={"station_id": str, "flag": str},
    keep_default_na=False,
)
assert len(observations) == station_manifest["rows"]
assert not observations.duplicated(["station_id", "date", "variable"]).any()
observations = observations[
    observations["station_id"].isin(
        pd.concat([evaluation_areas["station_id"], final_areas["station_id"]])
    )
].copy()
observations["date"] = pd.to_datetime(observations["date"])
assert observations["variable"].isin(["tmax", "tmin", "pcpn"]).all()
assert observations.loc[observations.variable.eq("pcpn"), "unit"].eq("inch").all()
assert observations.loc[~observations.variable.eq("pcpn"), "unit"].eq("degreeF").all()
observations["usable"] = pd.to_numeric(observations["value"], errors="coerce")
accepted = observations["flag"].str.strip().isin(["", "T"])
observations.loc[~accepted | ~np.isfinite(observations["usable"]), "usable"] = np.nan
weather_daily = observations.pivot(
    index=["station_id", "date"], columns="variable", values="usable"
).reset_index()
weather_daily["tmax_c"] = (weather_daily["tmax"] - 32) * 5 / 9
weather_daily["tmin_c"] = (weather_daily["tmin"] - 32) * 5 / 9
weather_daily["precip_mm"] = weather_daily["pcpn"] * 25.4
weather_daily["complete_weather"] = (
    np.isfinite(weather_daily[WEATHER]).all(axis=1)
    & weather_daily["precip_mm"].ge(0)
    & weather_daily["tmin_c"].le(weather_daily["tmax_c"])
)
weather_daily = weather_daily[["station_id", "date", *WEATHER, "complete_weather"]]

print(
    "Observed weather extends through",
    weather_daily.date.max().date(),
    "independently of activity coverage",
)

### 8. Aggregate nhãn daily cho hai catalogs

Pass thứ hai đếm trực tiếp group/day cho mỗi cohort. Tạo đủ daily keys **chỉ
trong training source window**. Target = recorded check-ins / fixed cohort size.
Mọi model dùng cùng key có weather đủ, including ngày không có recorded activity.


In [ ]:
evaluation_business_groups = (
    evaluation_cohort.groupby("business_id")["group_id"].agg(list).to_dict()
)
final_business_groups = final_cohort.groupby("business_id")["group_id"].agg(list).to_dict()
evaluation_daily, final_daily = Counter(), Counter()
end_label = FINAL_END.strftime("%Y-%m-%d")
with paths["checkin"].open() as handle:
    for line in handle:
        item = json.loads(line)
        final_groups = final_business_groups.get(item["business_id"])
        if final_groups is None:
            continue
        eval_groups = evaluation_business_groups.get(item["business_id"], ())
        for timestamp in item["date"].split(", "):
            day = timestamp[:10]
            if start_label <= day <= end_label:
                for group in eval_groups:
                    evaluation_daily[(group, day)] += 1
                for group in final_groups:
                    final_daily[(group, day)] += 1


def add_calendar(frame):
    frame = frame.copy()
    frame["calendar_month"] = frame["date"].dt.month
    frame["day_of_week"] = frame["date"].dt.dayofweek
    return frame


def build_panel(area_table, daily_counts, catalog):
    index = pd.MultiIndex.from_product(
        [area_table.index, pd.date_range(START, FINAL_END, freq="D").strftime("%Y-%m-%d")],
        names=["group_id", "date"],
    )
    counts = pd.Series(daily_counts)
    frame = counts.reindex(index, fill_value=0).rename("checkins").reset_index()
    assert frame.checkins.sum() == sum(daily_counts.values())
    frame["date"] = pd.to_datetime(frame["date"])
    frame = frame.merge(
        area_table.reset_index()[[*LOCATION_COLUMNS, "n_businesses"]],
        on="group_id",
        how="left",
        validate="many_to_one",
    )
    frame["rate"] = frame["checkins"] / frame["n_businesses"]
    frame = frame.merge(
        weather_daily, on=["station_id", "date"], how="left", validate="many_to_one"
    )
    frame["complete_weather"] = frame["complete_weather"].fillna(False).astype(bool)
    for column in [
        "group_id",
        "area_id",
        "metro",
        "city",
        "state",
        "zip",
        "category",
        "station_id",
    ]:
        frame[column] = frame[column].astype("category")
    frame = add_calendar(frame)
    assert not frame.duplicated(["group_id", "date"]).any()
    coverage = (
        frame.groupby([frame.date.dt.year, "metro", "category"], observed=True)
        .agg(
            rows=("date", "size"),
            complete_rows=("complete_weather", "sum"),
            checkins=("checkins", "sum"),
            zero_activity_share=("checkins", lambda x: x.eq(0).mean()),
        )
        .reset_index()
        .rename(columns={"date": "year"})
    )
    coverage["catalog"] = catalog
    coverage["weather_usable_share"] = coverage["complete_rows"] / coverage["rows"]
    return frame, coverage


evaluation_panel, eval_coverage = build_panel(evaluation_areas, evaluation_daily, "evaluation")
del evaluation_daily
gc.collect()
print(
    f"Evaluation panel: {len(evaluation_panel):,} keys; "
    f"{evaluation_panel.complete_weather.sum():,} weather-usable"
)
display(eval_coverage[eval_coverage.year.ge(VALIDATION_START.year)].round(4))

### 9. HGB pipeline, baseline và features tách khỏi labels

| Phần | Chi tiết |
| --- | --- |
| X | metro, industry, month/weekday, coordinates, tmax/tmin/rain |
| y | daily check-ins / historical cohort businesses |
| Weight | cohort size; không đi vào X |
| HGB | Poisson loss, 80 iterations, 15 leaves, seed42, no random early stopping |
| Baseline | training mean rate theo area/category/month |

Numeric scaling và one-hot được fit trong Pipeline chỉ bằng training. Mọi lần
predict đều chọn whitelist features riêng; nhãn chỉ dùng sau prediction để chấm.


In [ ]:
def feature_frame(frame):
    x = frame[FEATURES].copy()
    assert not FORBIDDEN_INPUTS.intersection(x.columns)
    assert np.isfinite(x[NUM_FEATURES]).all().all()
    return x


def fit_hgb(training):
    x_train = feature_frame(training)
    preprocessing = ColumnTransformer(
        [
            ("numeric", StandardScaler(), NUM_FEATURES),
            ("category", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_FEATURES),
        ],
        remainder="drop",
    )
    model = make_pipeline(preprocessing, HistGradientBoostingRegressor(**HGB_PARAMS))
    model.fit(
        x_train,
        training["rate"],
        histgradientboostingregressor__sample_weight=training["n_businesses"],
    )
    assert list(model.feature_names_in_) == FEATURES
    return model


def fit_baseline(training):
    return (
        training.groupby(BASELINE_KEYS, observed=True)["rate"]
        .mean()
        .rename("mean_rate")
        .reset_index()
    )


def predict_pair(model, baseline, evaluation):
    x = feature_frame(evaluation)
    model_prediction = np.maximum(model.predict(x), 1e-8)
    baseline_prediction = (
        evaluation[BASELINE_KEYS]
        .merge(baseline, on=BASELINE_KEYS, how="left", validate="many_to_one")["mean_rate"]
        .to_numpy()
    )
    assert np.isfinite(model_prediction).all() and np.isfinite(baseline_prediction).all()
    return {"hgb_weather": model_prediction, "calendar_mean": np.maximum(baseline_prediction, 1e-8)}


def evaluate_pair(model, baseline, evaluation, period):
    predictions = predict_pair(model, baseline, evaluation)
    tables, rows = [], []
    for name, prediction in predictions.items():
        result = evaluation[
            ["group_id", "area_id", "metro", "category", "date", "n_businesses", "checkins", "rate"]
        ].copy()
        result["predicted_rate"] = prediction
        result["model"], result["period"] = name, period
        tables.append(result)
        for (metro, category), segment in result.groupby(["metro", "category"], observed=True):
            rows.append(
                {
                    "period": period,
                    "model": name,
                    "metro": metro,
                    "category": category,
                    "poisson_deviance_rate": mean_poisson_deviance(
                        segment.rate, segment.predicted_rate, sample_weight=segment.n_businesses
                    ),
                    "mae_rate": mean_absolute_error(segment.rate, segment.predicted_rate),
                    "rows": len(segment),
                    "areas": segment.group_id.nunique(),
                    "actual_mean_rate": np.average(segment.rate, weights=segment.n_businesses),
                    "predicted_mean_rate": np.average(
                        segment.predicted_rate, weights=segment.n_businesses
                    ),
                }
            )
    segments = pd.DataFrame(rows)
    macro = (
        segments.groupby(["period", "model"])[["poisson_deviance_rate", "mae_rate"]]
        .mean()
        .reset_index()
    )
    return macro, segments, pd.concat(tables, ignore_index=True)


model_panel = evaluation_panel[evaluation_panel.complete_weather].copy()
initial_training = model_panel[model_panel.date.le(INITIAL_CUTOFF)]
validation = model_panel[model_panel.date.between(VALIDATION_START, VALIDATION_END)]
assert initial_training.date.max() < validation.date.min()
validation_model = fit_hgb(initial_training)
validation_baseline = fit_baseline(initial_training)
validation_metrics, validation_segments, validation_predictions = evaluate_pair(
    validation_model, validation_baseline, validation, VALIDATION_PERIOD
)
display(validation_metrics.round(6))
print("HGB is the preselected weather model; baseline remains the comparison, even if it wins.")

## Results

### 10. Test 2021 sau khi refit đến hết 2020, trước final fit

Refit evaluation model trên 2010–2020; giữ parameters và evaluation cohort
được chọn đến hết 2019. Predict toàn bộ ngày 2021 có weather hợp lệ trước khi
dùng nhãn 2021 để chấm. Không đọc activity của ngày trước hoặc rolling update.

Validation 2020 và test 2021 giúp kiểm tra giai đoạn gần cuối dữ liệu, có
distribution shift. Hai năm này đã được xem trong stress test trước; kết quả
là backtest khám phá, không phải accuracy của model refit đến 2022 hay của
weather forecast operational. Không chọn lại parameters dựa vào test.


In [ ]:
backtest_training = model_panel[model_panel.date.lt(TEST_START)]
backtest_model = fit_hgb(backtest_training)
backtest_baseline = fit_baseline(backtest_training)
metric_tables, segment_tables, prediction_tables = (
    [validation_metrics],
    [validation_segments],
    [validation_predictions],
)
evaluation = model_panel[model_panel.date.between(TEST_START, TEST_END)]
assert backtest_training.date.max() < evaluation.date.min()
assert validation.date.max() < evaluation.date.min()
metrics, segments, predictions = evaluate_pair(
    backtest_model, backtest_baseline, evaluation, TEST_PERIOD
)
metric_tables.append(metrics)
segment_tables.append(segments)
prediction_tables.append(predictions)
evaluation_metrics = pd.concat(metric_tables, ignore_index=True)
evaluation_segments = pd.concat(segment_tables, ignore_index=True)
evaluation_predictions = pd.concat(prediction_tables, ignore_index=True)
display(evaluation_metrics.round(6))
display(evaluation_segments[evaluation_segments.period.eq(TEST_PERIOD)].round(6))
# All model predictions precede truth scoring, and input mutation cannot affect predictions.
probe = validation.head(500).copy()
expected = validation_model.predict(feature_frame(probe))
for column in FORBIDDEN_INPUTS:
    probe[column] = 999999
assert np.allclose(validation_model.predict(feature_frame(probe)), expected)
assert np.allclose(
    validation_model.predict(
        feature_frame(probe.drop(columns=list(FORBIDDEN_INPUTS), errors="ignore"))
    ),
    expected,
)
for (_, _name), predicted in evaluation_predictions.groupby(["period", "model"], observed=True):
    assert (
        not predicted.duplicated(["group_id", "date"]).any()
        and predicted.predicted_rate.gt(0).all()
    )
print("Feature-only prediction and held-out key checks passed.")

### 11. Figures để đưa vào báo cáo

Deviance/MAE càng thấp càng tốt. Weather missingness thay đổi theo năm nên
report coverage cùng metrics. Monthly averages dưới đây chỉ dùng visualization,
không trở thành predictors; không suy ra causal weather effect.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, grouped in evaluation_metrics.groupby("model"):
    grouped = grouped.set_index("period").reindex([VALIDATION_PERIOD, TEST_PERIOD])
    axes[0].plot(range(len(grouped)), grouped.poisson_deviance_rate, marker="o", label=name)
    axes[1].plot(range(len(grouped)), grouped.mae_rate, marker="o", label=name)
for axis in axes:
    axis.set_xticks(range(2), [f"{VALIDATION_START.year} validation", f"{TEST_START.year} test"])
    axis.set_ylim(bottom=0)
    axis.legend()
axes[0].set(title="Historical conditional prediction", ylabel="Macro rate Poisson deviance")
axes[1].set(title="Held-out daily error", ylabel="MAE recorded check-ins / cohort business / day")
fig.tight_layout()
fig.savefig(RUN_DIR / "figures/evaluation.png", dpi=160)
plt.show()

selected = evaluation_predictions[
    evaluation_predictions.metro.eq("Tampa")
    & evaluation_predictions.category.eq("Restaurants")
    & evaluation_predictions.model.eq("hgb_weather")
].copy()
selected["month"] = selected.date.dt.to_period("M").dt.to_timestamp()
monthly = selected.groupby("month", observed=True).agg(
    actual_rate=("rate", "mean"), predicted_rate=("predicted_rate", "mean")
)
fig, axis = plt.subplots(figsize=(11, 3.5))
monthly.plot(ax=axis)
axis.set(
    title="Tampa Restaurants: held-out monthly mean of daily rates",
    ylabel="Recorded check-ins / cohort business / day",
    ylim=(0, None),
)
fig.tight_layout()
fig.savefig(RUN_DIR / "figures/held_out_activity.png", dpi=160)
plt.show()

### 12. Freeze evaluation outputs

Lưu evaluation catalog/labels/predictions và metrics trước khi train final.
Sau đó giải phóng panel evaluation để không giữ hai daily panels lớn cùng lúc.
File CSV đủ để tự recompute metrics và đối chiếu keys/coverage.


In [ ]:
evaluation_areas.reset_index().to_csv(RUN_DIR / "evaluation_areas.csv", index=False)
evaluation_panel[
    ["group_id", "date", "n_businesses", "checkins", "rate", *WEATHER, "complete_weather"]
].to_csv(RUN_DIR / "evaluation_labels.csv", index=False)
evaluation_metrics.to_csv(RUN_DIR / "evaluation_metrics.csv", index=False)
evaluation_segments.to_csv(RUN_DIR / "evaluation_segments.csv", index=False)
evaluation_predictions.to_csv(RUN_DIR / "evaluation_predictions.csv", index=False)
cohort_comparison.to_csv(RUN_DIR / "cohort_comparison.csv", index=False)
funnel.to_csv(RUN_DIR / "cohort_funnel.csv", index=False)
# The final-fit model is separate from all models used above for scoring.
del evaluation_panel, model_panel, initial_training, validation, backtest_training
# Keep only metric/report tables; large prediction frames can be reloaded from CSV.
del evaluation_predictions, validation_predictions, prediction_tables, selected, evaluation
# Models scored above remain artifacts only for this running research session.
del validation_model, backtest_model, validation_baseline, backtest_baseline
gc.collect()
print("Historical evaluation frozen before final training.")

### 13. Train HGB trên toàn bộ usable history với final catalog

Bao gồm tất cả daily keys tới source cutoff trong catalog cuối, không sample.
Refit preprocessing lẫn HGB bằng full history sau evaluation. Calendar baseline
cũng fit lại để UI có benchmark cùng catalog. Không báo training-fit error như
forecast accuracy; các holdout numbers ở trên thuộc evaluation catalog riêng.


In [ ]:
final_panel, final_coverage = build_panel(final_areas, final_daily, "final")
del final_daily
gc.collect()
full_training = final_panel[final_panel.complete_weather].copy()
assert full_training.date.min() == START
assert full_training.date.max() <= FINAL_END
final_model = fit_hgb(full_training)
final_baseline = fit_baseline(full_training)
final_training_summary = (
    full_training.groupby(["metro", "category"], observed=True)
    .agg(
        area_category_groups=("group_id", "nunique"),
        rows=("date", "size"),
        first_day=("date", "min"),
        last_day=("date", "max"),
        checkins=("checkins", "sum"),
        mean_rate=("rate", "mean"),
    )
    .reset_index()
)
annual_coverage = pd.concat([eval_coverage, final_coverage], ignore_index=True)
annual_coverage.to_csv(RUN_DIR / "annual_coverage.csv", index=False)
final_training_summary.to_csv(RUN_DIR / "final_training_summary.csv", index=False)
final_areas.reset_index().to_csv(RUN_DIR / "final_areas.csv", index=False)
final_cohort[["business_id", "group_id", "category"]].to_csv(
    RUN_DIR / "final_cohort.csv", index=False
)
final_panel[
    ["group_id", "date", "n_businesses", "checkins", "rate", *WEATHER, "complete_weather"]
].to_csv(RUN_DIR / "final_training_labels.csv", index=False)
print(
    f"Final HGB: {len(full_training):,}/{len(final_panel):,} daily rows; "
    f"{len(final_areas):,} area/category groups"
)
display(final_training_summary.assign(mean_rate=lambda frame: frame.mean_rate.round(5)))

### 14. Export bundle tương thích với Forecast UI

Bundle chỉ chứa trained Pipeline và input metadata; không cần labels/activity
history để predict. Joblib chỉ được load từ bundle do notebook local này tạo;
loader kiểm tra hashes, schema và dependency versions trước deserialize. Không
nhận model upload từ người dùng. Giữ versions bằng `uv.lock`.


In [ ]:
ARTIFACT_DIR = RUN_DIR / "artifact"
ARTIFACT_DIR.mkdir()
locations = final_areas.reset_index()[LOCATION_COLUMNS].copy()
assert not FORBIDDEN_INPUTS.intersection(locations.columns)
joblib.dump(final_model, ARTIFACT_DIR / "model.joblib", compress=3)
locations.to_csv(ARTIFACT_DIR / "locations.csv", index=False)
final_baseline.to_csv(ARTIFACT_DIR / "baseline.csv", index=False)
weather_daily.to_csv(ARTIFACT_DIR / "weather_daily.csv", index=False)
artifact_files = ["model.joblib", "locations.csv", "baseline.csv", "weather_daily.csv"]
weather_range = {
    column: {"min": float(full_training[column].min()), "max": float(full_training[column].max())}
    for column in WEATHER
}
artifact_manifest = {
    "schema_version": 1,
    "status": "complete",
    "model_name": "hgb_weather",
    "created_at_utc": datetime.now(UTC).isoformat(),
    "features": FEATURES,
    "versions": {
        key: importlib.metadata.version(package)
        for key, package in {
            "sklearn": "scikit-learn",
            "numpy": "numpy",
            "pandas": "pandas",
            "joblib": "joblib",
        }.items()
    },
    "trained_start": str(full_training.date.min().date()),
    "trained_end": str(full_training.date.max().date()),
    "cohort_selection_end": str(FINAL_END.date()),
    "final_catalog": final_funnel,
    "evaluation_catalog": eval_funnel,
    "evaluation_cohort_selection_end": str(INITIAL_CUTOFF.date()),
    "evaluation_metrics": evaluation_metrics.to_dict(orient="records"),
    "weather_range": weather_range,
    "hyperparameters": HGB_PARAMS,
    "source_hashes": source_hashes,
    "notebook_sha256": sha256(SOURCE_NOTEBOOK),
    "lockfile_sha256": sha256(ROOT / "uv.lock"),
    "source_boundary_audit": source_audit_counts,
    "artifact_sha256": {name: sha256(ARTIFACT_DIR / name) for name in artifact_files},
    "evaluation_policy": EVALUATION_POLICY,
    "evaluation_protocol": EVALUATION_PROTOCOL,
    "timestamp_policy": "Source date labels; timezone and station windows unverified",
    "prediction_interpretation": "Daily check-ins/cohort business; no 2026 activity labels",
}
(ARTIFACT_DIR / "manifest.json").write_text(
    json.dumps(artifact_manifest, ensure_ascii=False, indent=2) + "\n"
)
bundle = load_bundle(ARTIFACT_DIR)
assert list(bundle.model.feature_names_in_) == FEATURES
print("Runtime bundle passed hash/schema/version validation.")

### 15. Predict qua code dùng trong UI và kiểm tra invariants

Có thể predict ngày bất kỳ với weather nhập. Observed 2026 lấy đúng daily report
ở station của candidate, report exclusions. Baseline và HGB là predictions từ
train; chưa có actual activity 2026 để đo accuracy. File mẫu phục vụ demo/report.


In [ ]:
sample_specifications = [
    ("philadelphia_observed", "2026-07-15", "Philadelphia", "Restaurants", None),
    ("nashville_observed", "2026-01-15", "Nashville", "Restaurants", None),
    ("tampa_spa_observed", "2026-09-15", "Tampa", "Day Spas", None),
    (
        "future_scenario",
        "2026-12-15",
        "Tampa",
        "Day Spas",
        {"tmax_c": 27, "tmin_c": 19, "precip_mm": 2},
    ),
]
samples = []
for label, day, metro, category, weather in sample_specifications:
    results = predict_locations(bundle, pd.Timestamp(day).date(), metro, category, weather)
    results["sample"] = label
    samples.append(results)
    display(
        results.head(3)[
            [
                "date",
                "metro",
                "category",
                "city",
                "zip",
                "predicted_rate",
                "baseline_rate",
                *WEATHER,
            ]
        ].round(5)
    )
sample_predictions = pd.concat(samples, ignore_index=True)
sample_predictions.to_csv(RUN_DIR / "sample_predictions.csv", index=False)
assert not sample_predictions.duplicated(["sample", "group_id", "date"]).any()
assert not FORBIDDEN_INPUTS.intersection(sample_predictions.columns)
for day in ["2027-05-01", "2028-02-29"]:
    future = predict_locations(
        bundle,
        pd.Timestamp(day).date(),
        "Tampa",
        "Day Spas",
        {"tmax_c": 27, "tmin_c": 19, "precip_mm": 2},
    )
    assert not future.empty and future.date.eq(pd.Timestamp(day).date()).all()
reference = predict_locations(
    bundle,
    pd.Timestamp("2026-12-15").date(),
    "Tampa",
    "Day Spas",
    {"tmax_c": 27, "tmin_c": 19, "precip_mm": 2},
)
# Extra activity metadata cannot reach Pipeline.predict because runtime whitelists FEATURES.
for column in FORBIDDEN_INPUTS:
    bundle.locations[column] = 999999
try:
    assert np.allclose(
        predict_locations(
            bundle,
            pd.Timestamp("2026-12-15").date(),
            "Tampa",
            "Day Spas",
            {"tmax_c": 27, "tmin_c": 19, "precip_mm": 2},
        ).predicted_rate,
        reference.predicted_rate,
    )
finally:
    bundle.locations.drop(columns=list(FORBIDDEN_INPUTS), inplace=True)
print("Runtime predictions: future/leap date, source coverage, label isolation checks passed.")

### 16. Publish local model cho UI sau khi bundle đã kiểm tra

Copy từng artifact bằng replace, ghi manifest cuối để loader không nhận bundle
nửa chừng. App cache theo manifest revision. Đây là publish local file, không
push model lên Git hoặc deploy server. Model folder đã ignored; nếu chạy container
thì mount/copy bundle tương ứng tại `/app/models/location_forecast`.


In [ ]:
MODEL_DIR = ROOT / "models/location_forecast"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
for filename in [*artifact_files, "manifest.json"]:
    destination = MODEL_DIR / filename
    temporary = destination.with_name(destination.name + ".tmp")
    shutil.copyfile(ARTIFACT_DIR / filename, temporary)
    temporary.replace(destination)
published = load_bundle(MODEL_DIR)
published_result = predict_locations(
    published,
    pd.Timestamp("2026-12-15").date(),
    "Tampa",
    "Day Spas",
    {"tmax_c": 27, "tmin_c": 19, "precip_mm": 2},
)
assert np.allclose(published_result.predicted_rate, reference.predicted_rate)
print("Forecast UI bundle ready:", MODEL_DIR.relative_to(ROOT))
print("Open Demand forecast in the current Streamlit app; choose date/weather and Predict.")

### 17. Report và provenance manifest

Generated report gồm phạm vi, split, metrics, missingness, final fit và limitations.
Figures/CSV có thể đưa vào bài báo cáo; bản executed notebook giữ toàn bộ tables.
Manifest hashes giúp kiểm tra nguồn/model/report sau này.


In [ ]:
test_metrics = evaluation_metrics[evaluation_metrics.period.eq(TEST_PERIOD)]
metrics_markdown = "| Period | Model | Rate deviance | Rate MAE |\n| --- | --- | ---: | ---: |\n"
for row in evaluation_metrics.itertuples(index=False):
    metrics_markdown += (
        f"| {row.period} | {row.model} | {row.poisson_deviance_rate:.6f} | {row.mae_rate:.6f} |\n"
    )
excluded_before = source_audit_counts["global_events_before_start"]
excluded_last = source_audit_counts["global_events_on_excluded_last_day"]
new_groups = len(set(final_areas.index) - set(evaluation_areas.index))
report = f"""# SiteSense HGB full-history training report

Generated UTC: {datetime.now(UTC).isoformat()}

Scope: Restaurants and Day Spas, county groups Philadelphia/Nashville/Tampa.
Full source JSONL files are read; no random sampling.
Activity source: {source_first} to {source_last}.
Fit window: {START.date()} to {FINAL_END.date()}.
Boundary policy: exclude {excluded_before} sparse 2009 events and final source day
({excluded_last} global events). Retain January 2022 completed dates through {FINAL_END.day}.

Evaluation catalog: {len(evaluation_areas)} area/category groups;
{len(evaluation_cohort)} business/category rows, selected through {INITIAL_CUTOFF.date()}.
Initial train: {START.date()} to {INITIAL_CUTOFF.date()}.
Validation: {VALIDATION_START.date()} to {VALIDATION_END.date()}.
Refit through {VALIDATION_END.date()}; test: {TEST_START.date()} to {TEST_END.date()}.
Evaluation catalog stays fixed at the pre-validation cutoff.
Evaluation uses observed target-day weather. The 2020/2021 labels were viewed
in the previous stress run: this is an exploratory backtest, not an untouched
confirmatory test. Hyperparameters remain fixed from the original POC.

{metrics_markdown}

Final model: HGB Poisson, 80 iterations, 15 leaves, seed 42.
Fit {len(full_training):,} weather-usable rows of {len(final_panel):,} daily labels.
Final catalog: {len(final_areas)} area/category groups; {len(final_cohort)} business/category rows.
{new_groups} groups added after evaluation.
Excluded final rows for missing/invalid weather: {len(final_panel) - len(full_training):,}.
See annual_coverage.csv for changing coverage and source-date assumptions.

Prediction inputs:
{", ".join(FEATURES)}
Activity, exposure, lag and trend are excluded from prediction inputs.
Target: recorded check-ins per fixed historical cohort business per day.

Artifacts: models/location_forecast/
model.joblib, locations.csv, weather_daily.csv, baseline.csv, manifest.json.
Figures: figures/evaluation.png and figures/held_out_activity.png.

Limitations: Yelp selection and snapshot inventory; no historical opening/closing
dates; source-date alignment unverified; observed-weather conditional prediction,
not weather forecasting; no actual 2026 activity labels; no new-store-success,
revenue or causality claims. Baseline may beat HGB; retain both metrics.
Historical metrics describe the historical evaluation catalog; accuracy of the
expanded final training catalog has not been measured out of sample.
"""
(RUN_DIR / "report.md").write_text(report)
output_paths = [
    path for path in RUN_DIR.rglob("*") if path.is_file() and path.name != "manifest.json"
]
run_manifest = {
    "status": "complete",
    "created_at_utc": datetime.now(UTC).isoformat(),
    "purpose": "Full-history HGB training, temporal evaluation and local UI artifact",
    "source_paths": {key: str(value.relative_to(ROOT)) for key, value in paths.items()},
    "source_hashes": source_hashes,
    "source_audit": source_audit_counts,
    "notebook_sha256": sha256(SOURCE_NOTEBOOK),
    "lockfile_sha256": sha256(ROOT / "uv.lock"),
    "category_parser_sha256": sha256(ROOT / "src/sitesense/import_data/yelp.py"),
    "runtime_sha256": sha256(ROOT / "src/sitesense/forecast.py"),
    "versions": versions,
    "features": FEATURES,
    "hyperparameters": HGB_PARAMS,
    "evaluation_catalog": eval_funnel,
    "final_catalog": final_funnel,
    "final_training_rows": len(full_training),
    "final_panel_rows": len(final_panel),
    "final_training_end": str(FINAL_END.date()),
    "final_model_directory": str(MODEL_DIR.relative_to(ROOT)),
    "evaluation_metrics": evaluation_metrics.to_dict(orient="records"),
    "evaluation_policy": EVALUATION_POLICY,
    "evaluation_protocol": EVALUATION_PROTOCOL,
    "output_sha256": {str(path.relative_to(RUN_DIR)): sha256(path) for path in output_paths},
}
(RUN_DIR / "manifest.json").write_text(
    json.dumps(run_manifest, ensure_ascii=False, indent=2) + "\n"
)
display(test_metrics.round(6))
print(report)
print("All steps completed. Report and evidence:", RUN_DIR.relative_to(ROOT))

## Takeaways

- Report historical out-of-sample metrics trước khi mô tả final fit. Không dùng
  training-fit score làm accuracy hay gán metric của catalog cũ cho catalog mới.
- HGB dùng đúng input weather/location/industry/date. Bỏ activity history vẫn
  predict ngày bất kỳ với weather hợp lệ; nhãn chỉ tham gia train và scoring.
- Bundle thật được load/predict bằng `sitesense.forecast`, cùng path code UI.
  Demand forecast dùng artifact, không query activity DB hoặc train lúc render.
- Giữ baseline và ghi weather missingness/shift trong báo cáo. Chưa có actual
  2026 để đo accuracy2026, hoặc dữ liệu đánh giá thành công của cửa hàng mới.
- Đọc generated `report.md`, figures và tables của lần chạy này để lấy số liệu.

Bản final fit sử dụng toàn bộ weather-usable daily rows đến 2022-01-18.
Đọc coverage/catalog trong report để phân biệt phạm vi evaluation và final fit.
Khi so sánh với lần chạy cũ, lưu ý cutoff training và evaluation catalog đã đổi;
so sánh HGB với baseline trên cùng holdout/catalog của lần chạy mới là trực tiếp.
